# Canonical Correlation Analysis (CCorA)

## Overview
Canonical Correlation Analysis (CCorA) examines the relationships between **two sets of variables** measured on the same sampling units. It finds pairs of linear combinations — one from each set — that are maximally correlated.

**Key distinction from other methods:**

| Method | Question | Response structure |
|---|---|---|
| Multiple regression | How do Xs predict one Y? | One Y, many Xs |
| MANOVA/LDA | Do groups differ on many Ys? | Many Ys, one categorical X |
| RDA | How do env variables drive species? | Many Ys, many Xs (asymmetric) |
| **CCorA** | How are two variable sets related? | Many Ys ↔ Many Xs (symmetric) |

**Ecological examples:** morphometric measurements vs. life-history traits; physical vs. chemical environment variables; diet composition vs. physiology.

**Naming caution:** in ecology, "CCA" almost always means Canonical Correspondence Analysis (a constrained ordination for species data). CCorA is a different technique entirely. This notebook covers CCorA.

**Reference:** Quinn & Keough (2002) ch. 17

---

In [ ]:
library(tidyverse); library(CCA); library(vegan)
set.seed(31)
# 50 fish: Set 1 = morphology (4 vars), Set 2 = physiology (3 vars)
# True relationship: large fish have higher metabolic rates
n <- 50
size_factor <- rnorm(n, 0, 1)

morphology <- data.frame(
  total_length  = 25 + 4 * size_factor + rnorm(n, 0, 0.8),
  body_depth    = 8  + 1.5 * size_factor + rnorm(n, 0, 0.5),
  head_width    = 5  + 1.2 * size_factor + rnorm(n, 0, 0.4),
  tail_length   = 6  + 0.8 * size_factor + rnorm(n, 0, 0.6)
)

physiology <- data.frame(
  metabolic_rate  = 80 + 10 * size_factor + rnorm(n, 0, 3),
  gill_area       = 12 +  2 * size_factor + rnorm(n, 0, 1),
  oxygen_uptake   = 5  +  1 * size_factor + rnorm(n, 0, 0.8)
)

cat("Dataset: n =", n, "fish\n")
cat("Set 1 (morphology):", ncol(morphology), "variables\n")
cat("Set 2 (physiology):", ncol(physiology), "variables\n")

# Scale both sets
X <- scale(morphology)
Y <- scale(physiology)

---
## Correlation structure between sets

In [ ]:
# Examine between-set correlations
cat("--- Cross-correlation matrix (morphology vs physiology) ---\n")
cross_cor <- cor(X, Y)
print(round(cross_cor, 3))
cat("\nWithin-set correlations for X (morphology):\n")
print(round(cor(X), 3))
cat("\nWithin-set correlations for Y (physiology):\n")
print(round(cor(Y), 3))

In [ ]:
# Canonical Correlation Analysis using CCA package
cc_fit <- cc(X, Y)

cat("=== Canonical correlations ===\n")
for (i in seq_along(cc_fit$cor))
  cat(sprintf("  Pair %d: r_c = %.4f (r² = %.4f)\n",
              i, cc_fit$cor[i], cc_fit$cor[i]^2))

# Canonical loadings (structure coefficients)
cat("\n--- Canonical loadings: Set X (morphology) ---\n")
print(round(cc_fit$scores$corr.X.xscores, 3))
cat("\n--- Canonical loadings: Set Y (physiology) ---\n")
print(round(cc_fit$scores$corr.Y.yscores, 3))
cat("(Loadings ≥ |0.30| typically interpreted as meaningful)\n")

---
## Significance testing and redundancy

In [ ]:
# Wilks' Lambda test for each canonical pair
# Tests H0: remaining canonical correlations are all zero
cat("--- Wilks' Lambda significance tests ---\n")
cc_test <- p.asym(cc_fit$cor, n, ncol(X), ncol(Y), tstat = "Wilks")
print(cc_test)
cat("Interpret sequentially: first test includes all pairs,\n")
cat("subsequent tests drop the largest pair one at a time.\n")

In [ ]:
# Redundancy analysis: proportion of variance in each set
# explained by the canonical variates of the OTHER set
cat("--- Redundancy index (Stewart-Love) ---\n")
cc_sum <- comput(X, Y, cc_fit)

# Variance explained in Y by X's canonical variates
prop_var_Y <- cc_sum$corr.Y.yscores^2  # squared loadings
prop_shared <- sweep(prop_var_Y, 2, cc_fit$cor^2, "*")
cat("Proportion of Y variance explained by each canonical variate:\n")
cat("  (colMeans of squared canonical loadings × r_c²)\n")
redund_Y <- colMeans(prop_var_Y) * cc_fit$cor^2
for (i in seq_along(redund_Y))
  cat(sprintf("  Pair %d: %.3f (%.1f%%)\n", i, redund_Y[i], redund_Y[i]*100))
cat("Total redundancy (Y given X):", round(sum(redund_Y), 3), "\n")

# Biplot of first canonical pair
scores_X <- cc_fit$scores$xscores
scores_Y <- cc_fit$scores$yscores
plot_df <- data.frame(
  CV1_X = scores_X[, 1], CV1_Y = scores_Y[, 1],
  CV2_X = scores_X[, 2], CV2_Y = scores_Y[, 2]
)
ggplot(plot_df, aes(CV1_X, CV1_Y)) +
  geom_point(alpha = 0.7, colour = "steelblue", size = 2.5) +
  geom_smooth(method = "lm", se = TRUE, colour = "firebrick", linewidth = 0.9) +
  labs(title = sprintf("First canonical pair: r_c = %.3f", cc_fit$cor[1]),
       x = "Canonical variate 1 (Morphology)",
       y = "Canonical variate 1 (Physiology)") +
  theme_minimal()

---
## Common Pitfalls

**1. Confusing CCorA with Canonical Correspondence Analysis**
In ecology, "CCA" invariably means Canonical Correspondence Analysis (chi-square-based constrained ordination for species data, available in `vegan::cca()`). CCorA is a different method for relating two continuous variable sets symmetrically. The naming confusion is pervasive — be explicit when writing up results.

**2. Interpreting canonical correlations without checking redundancy**
A high canonical correlation does not mean each set is well explained by the other. Redundancy (Stewart-Love index) measures the average variance in one set explained by the canonical variates derived from the other. A high r_c with low redundancy means only a small portion of each set is actually accounted for.

**3. Over-interpreting non-significant canonical pairs**
Only the first (or first few) canonical pairs are usually significant. Non-significant pairs represent noise. Use the sequential Wilks' lambda test and interpret only significant pairs.

**4. Using raw canonical coefficients instead of structure loadings**
Like in LDA, raw canonical coefficients are affected by multicollinearity within sets. Use canonical loadings (correlations between original variables and canonical variates) — these are more stable and interpretable.

**5. Applying CCorA with more variables than observations**
CCorA requires inverting within-set covariance matrices. With p + q ≥ n, this fails. Reduce dimensionality first (PCA, variable selection) before applying CCorA.

**6. Symmetric framing when the scientific question is asymmetric**
If you have a clear response set (e.g., species community) and predictor set (e.g., environmental variables), RDA is more appropriate than CCorA because it imposes the predictor-response asymmetry that matches the biological question. CCorA is appropriate when both variable sets are genuinely co-equal (e.g., morphology vs. physiology measured on the same individuals).


---
*r_methods_library - Samantha McGarrigle*